In [9]:
#NEWS Intelligence readers

In [21]:
!pip -q install -U google-genai feedparser
!pip install --upgrade google-auth

In [33]:
import google.genai as genai # Updated import
import google.auth

print(f"google-genai version: {genai.__version__}") # Updated print statement
print(f"google-auth version: {google.auth.__version__}")

google-genai version: 2.19.0
google-auth version: 2.56.3


In [34]:
import os, json, re
import feedparser         #Library helps for RSS related ops
from datetime import datetime
from time import sleep
from IPython.display import HTML, display
import google.genai as genai # Explicitly import google.genai for consistency

In [ ]:
# ----------------------------
# API KEY SETUP
# ----------------------------

# Set your API key as an environment variable
os.environ["GEMINI_API_KEY"] = "..." # <-- your gemini api key

API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Please set GEMINI_API_KEY as an environment variable")

client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.6-flash"

In [41]:
# ============================================================
# STEP 1: LLM HELPER FUNCTIONS
# ============================================================
"""
WHAT:
We define helper utilities to:
- Extract text safely from model responses
- Retry LLM calls
- Parse JSON safely from imperfect outputs

HOW:
- Carefully extract text from the response object
- Retry failed calls
- Use regex fallback for JSON parsing
"""

def extract_text(resp):
    """
    Extract text safely from Gemini response.
    """
    # TODO: If resp has a `.text` attribute, return .strip() of the text
    if hasattr(resp,"text") and resp.text:
      return resp.text.strip()

    try:
      parts = []
      for c in resp.candidates:
        for p in c.content.parts:
          if hasattr(p,"text") and p.text:
            parts.append(p.text)
      return "\n".join(parts).strip()
    except:
      return ""


def llm(prompt, retries=2):
    """
    Call the LLM with retries.
    """
    for _ in range(retries):
        try:
            # TODO: Call client.models.generate_content
            # TODO: Use extract_text to get response text
            # TODO: Return text if successful
            r = client.models.generate_content(
                model = MODEL_NAME,
                contents = prompt
            )
            print(f"DEBUG: Raw LLM response object: {r}") # Debugging print
            text = extract_text(r)
            print(f"DEBUG: Extracted text from response: '{text}'") # Debugging print
            if text:
              return text
            pass
        except Exception as e:
            print(f"DEBUG: Exception during LLM call: {e}") # Debugging print
            sleep(1)
    return ""


def safe_json(s):
    if not s:
        return None
    try:
        return json.loads(s)
    except:
        m = re.search(r"\{.*\}|\[.*\]", s, flags=re.DOTALL)
        return json.loads(m.group(0)) if m else None

In [42]:
RSS_FEEDS = {
    "BBC Technology": "http://feeds.bbci.co.uk/news/technology/rss.xml",
    "The Verge": "https://www.theverge.com/rss/index.xml",
    "TechCrunch": "https://techcrunch.com/feed/",
}

MAX_ITEMS_PER_FEED = 6

In [43]:

def fetch_rss(source, url):
  feed = feedparser.parse(url) #parse the rss feed
  items = []
  for e in feed.entries[:MAX_ITEMS_PER_FEED]:
    link = getattr(e,"link","").strip()
    title = getattr(e,"title","").strip()
    if title and link:
      items.append({
          "source":source,
           "title":title,
           "link":link,
           "published":getattr(e,"published","")
      }
      )
  return items

raw_items = []
for src,url in RSS_FEEDS.items():
  raw_items.extend(fetch_rss(src,url))

print(f"Fetched {len(raw_items)} raw headlines.")

raw_blob = "\n".join(
    f"{i+1}. source={it['source']} | title={it['title']} | link={it['link']}"
    for i, it in enumerate(raw_items)
)



Fetched 18 raw headlines.


In [47]:
call1_prompt = f"""

Clean and deduplicate news headlines

Ensure:
- Remove duplicates/ Near duplicates
- Normalize Titles
- Dont Summarize
- Output ONLY valid JSON Format

 Format:
 [
   {{
     "id": 1,
     "title": "...",
     "source": "...",
     "link": "...",
     "date": "..."
   }}
 ]

 Keep 8–15 distinct stories.
RAW:
{raw_blob}
"""
print(call1_prompt)
llm_output_raw = llm(call1_prompt)
clean_items = safe_json(llm_output_raw)

if clean_items is None:
    print(f"Warning: LLM output could not be parsed into JSON. Raw LLM output: {llm_output_raw}")
    clean_items = [] # Fallback to an empty list

assert isinstance(clean_items, list), "Call 1 failed"



Clean and deduplicate news headlines

Ensure:
- Remove duplicates/ Near duplicates
- Normalize Titles
- Dont Summarize
- Output ONLY valid JSON Format

 Format:
 [
   {
     "id": 1,
     "title": "...",
     "source": "...",
     "link": "...",
     "date": "..."
   }
 ]

 Keep 8–15 distinct stories.
RAW:
1. source=BBC Technology | title=Robot horse and rider steal the spotlight at Chinese conference | link=https://www.bbc.co.uk/news/videos/cy4k4d3lj21o?at_medium=RSS&at_campaign=rss
2. source=BBC Technology | title=TikTok to pay $400m to US in one of largest child privacy settlements | link=https://www.bbc.co.uk/news/articles/cwyr0l45xjro?at_medium=RSS&at_campaign=rss
3. source=BBC Technology | title=How landscape gardening is being electrified | link=https://www.bbc.co.uk/news/articles/cpq3w3v19veo?at_medium=RSS&at_campaign=rss
4. source=BBC Technology | title=Is Vine back? Short-form video-sharing app Divine opens to public | link=https://www.bbc.co.uk/news/articles/c3r05g378w4o?a

In [48]:
# ============================================================
# STEP 4: CALL 2 — SUMMARIZE NEWS ITEMS
# ============================================================

# TODO: Write summarization prompt
call2_prompt = f"""
Write short news briefs
For each item
 -1 paragraph (3-4 lines)
 -3 key bullet points
 -No invented facts

Output ONLY valid JSON.

Format:
[
  {{
    "id": 1,
    "title": "...",
    "source": "...",
    "link": "...",
    "summary": "...",
    "key_points": ["...", "...", "..."]
  }}
]

ITEMS:
{json.dumps(clean_items, ensure_ascii=False)}
"""

llm_output_raw_call2 = llm(call2_prompt)
news_cards = safe_json(llm_output_raw_call2)

if news_cards is None:
    print(f"Warning: LLM output for Call 2 could not be parsed into JSON. Raw LLM output: {llm_output_raw_call2}")
    news_cards = [] # Fallback to an empty list

assert isinstance(news_cards, list), "Call 2 failed"

DEBUG: Raw LLM response object: sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""[
  {
    "id": 1,
    "title": "TikTok to Pay $400M to US in One of Largest Child Privacy Settlements",
    "source": "BBC Technology",
    "link": "https://www.bbc.co.uk/news/articles/cwyr0l45xjro",
    "summary": "TikTok has agreed to pay a $400 million settlement to the United States following allegations regarding child privacy violations. The agreement represents one of the largest settlements of its kind involving young users' data protection. Authorities reached the deal after extensive regulatory scrutiny into how the popular video platform handles minor account data.",
    "key_points": [
      "TikTok agrees to pay $400 million in a major US child privacy settlement.",
      "The penalty stands among the largest ever imposed for child data protection violations.",
      "The deal resolves regulatory scruti

In [49]:
# ============================================================
# CALL 3: SCORE ONLY (NO SORTING)
# ============================================================
call3_prompt = f"""
Score each news item.

Definitions:
- Urgency (0–5)
- Impact (0–5)

Output ONLY valid JSON:

[
  {{
    "id": 1,
    "urgency": 0,
    "impact": 0,
    "reason": "one sentence"
  }}
]

ITEMS:
{json.dumps(news_cards, ensure_ascii=False)}
"""

scores = safe_json(llm(call3_prompt))
assert isinstance(scores, list), "Call 3 failed"

DEBUG: Raw LLM response object: sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""[
  {
    "id": 1,
    "urgency": 2,
    "impact": 4,
    "reason": "TikTok's $400 million settlement with the US represents one of the largest privacy penalties ever imposed involving minor user data."
  },
  {
    "id": 2,
    "urgency": 2,
    "impact": 2,
    "reason": "The public launch of video app Divine brings a new consumer entrant seeking to recreate the nostalgic format of Vine."
  },
  {
    "id": 3,
    "urgency": 3,
    "impact": 2,
    "reason": "A 1,500% price spike for essential invoicing software forces affected UK businesses to immediately re-evaluate software budgets."
  },
  {
    "id": 4,
    "urgency": 3,
    "impact": 3,
    "reason": "HMRC issuing 81,000 enforcement letters marks a significant regulatory crackdown on undeclared UK cryptocurrency gains."
  },
  {
    "id": 5,
    "urgency": 1,

In [52]:

#Sorting

cards = {c["id"]:c for c in news_cards if "id" in c}

score_map={}
for s in scores:
  if "id" in s:
    urgency = int(s["urgency"]) if "urgency" in s else 0
    impact = int(s["impact"]) if "impact" in s else 0

  score_map[s["id"]] = {
            "urgency": urgency,
            "impact": impact,
            "score": urgency + impact,
            "reason": s.get("reason", "")
        }

ranked = []
for _id, meta in score_map.items():
    if _id in cards:
        ranked.append({**cards[_id], **meta})

ranked.sort(
    key=lambda x: (x["score"], x["urgency"], x["impact"]),
    reverse=True
)
TOP_K = 5
top_news = ranked[:TOP_K]

In [53]:
# ============================================================
# HTML OUTPUT
# ============================================================
html = f"""
<style>
:root {{
  --border: color-mix(in srgb, currentColor 20%, transparent);
}}
.container {{
  max-width: 960px;
  margin: auto;
  font-family: system-ui, -apple-system, sans-serif;
}}
.header {{
  display: flex;
  justify-content: space-between;
  margin: 16px 0;
}}
.card {{
  border: 1px solid var(--border);
  border-radius: 14px;
  padding: 16px;
  margin: 14px 0;
}}
.badges {{
  display: flex;
  gap: 8px;
  flex-wrap: wrap;
  font-size: 12px;
}}
.badge {{
  padding: 4px 8px;
  border-radius: 999px;
  border: 1px solid var(--border);
}}
.title {{
  font-size: 18px;
  margin: 10px 0;
}}
.summary {{
  margin: 8px 0;
  line-height: 1.4;
}}
ul {{
  margin: 6px 0 0 18px;
}}
.footer {{
  opacity: 0.7;
  font-size: 13px;
  margin-top: 12px;
}}
</style>

<div class="container">
  <div class="header">
    <h2>🗞️ Top {TOP_K} Tech News Brief</h2>
    <div class="footer">{datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
  </div>
"""

for i, n in enumerate(top_news, 1):
    html += f"""
    <div class="card">
      <div class="badges">
        <span class="badge">Rank #{i}</span>
        <span class="badge">Score {n['score']}/10</span>
        <span class="badge">Urgency {n['urgency']}/5</span>
        <span class="badge">Impact {n['impact']}/5</span>
        <span class="badge">{n['source']}</span>
      </div>

      <div class="title">{n['title']}</div>
      <div class="summary">{n['summary']}</div>

      <ul>
        {''.join(f"<li>{kp}</li>" for kp in n['key_points'])}
      </ul>

      <div class="summary"><b>Why it matters:</b> {n['reason']}</div>

      <div class="footer">
        <a href="{n['link']}" target="_blank">Read original</a>
      </div>
    </div>
    """

html += "</div>"
display(HTML(html))
